# **Mumax⁺ in Google Colaboratory**

# About Google Colaboratory

Google Colaboratory is a research tool mainly used by researchers in the field of machine learning. The main purpose of this tool is to run Python code in Jupyter notebooks. These notebooks run on a virtual Linux machine private to your Gmail account. This means that you will need a Gmail account to execute programs in a Google Colaboratory session. If you have a Gmail account you should be able to copy this Jupyter notebook to your Google Drive and execute the code cells.

Run the cell below to connect to a GPU-enabled Colaboratory runtime and display its details.

In [ ]:
! echo "This machine runs" $(uname) "with" $(python --version) "and provides the following GPU/CUDA specs:"
! nvidia-smi

# Installing mumax⁺

To run mumax⁺ simulations, the `mumaxplus` Python package must be installed in this Colab environment. To this end, we will make our life easy by using the appropriate pre-built wheel provided in the mumax⁺ GitHub release assets. Running the cell below will install mumax⁺, which may take a minute to complete. When the installation is done, you can collapse this section to get a clean workspace.

Choose which version of mumax⁺ to install (&geq;1.2.0) in the cell below. For a list of versions/wheels, see the [GitHub releases](https://github.com/mumax/plus/releases).

In [ ]:
MUMAXPLUS_VERSION = "1.2.0"

try:
    import google.colab
    import requests
except ImportError:
    pass
else:
    # Determine which wheel to use
    from platform import python_version_tuple
    major, minor, patch = python_version_tuple()
    PYVERSION = major + minor
    # Download the mumax+ wheel
    URL = f"https://github.com/mumax/plus/releases/download/v{MUMAXPLUS_VERSION}/mumaxplus-{MUMAXPLUS_VERSION}-cp{PYVERSION}-cp{PYVERSION}-manylinux_2_34_x86_64.whl"
    if requests.head(URL).status_code not in [200, 302]:
        raise ValueError(f"No wheel found for mumax+ {MUMAXPLUS_VERSION} on Python {major}.{minor}. Try another mumax+ version.")
    else:
        cmd = f'pip install -v "{URL}"'
        !{cmd}

# Running mumax+

Now it's up to you! The example below solves Standard Problem 4 using mumax⁺, but feel free to create a copy of this notebook to solve another micromagnetic problem of your choice.

In [ ]:
# This script solves micromagnetic Standard Problem 4. The Problem specification
# can be found on https://www.ctcms.nist.gov/~rdm/mumag.org.html

import matplotlib.pyplot as plt
import numpy as np

from mumaxplus import Ferromagnet, Grid, World

length, width, thickness = 500e-9, 125e-9, 3e-9
nx, ny, nz = 128, 32, 1
world = World(cellsize=(length / nx, width / ny, thickness / nz))

magnet = Ferromagnet(world, Grid((nx, ny, nz)))
magnet.msat = 800e3
magnet.aex = 13e-12
magnet.alpha = 0.02

magnet.magnetization = (1, 0.1, 0)
magnet.minimize()

B1 = (-24.6e-3, 4.3e-3, 0)
B2 = (-35.5e-3, -6.3e-3, 0)
world.bias_magnetic_field = B1  # choose B1 or B2 here

# --- SCHEDULE THE OUTPUT ---
timepoints = np.linspace(0, 1e-9, 1000)
outputquantities = {
    "mx": lambda: magnet.magnetization.average()[0],
    "my": lambda: magnet.magnetization.average()[1],
    "mz": lambda: magnet.magnetization.average()[2],
    "e_total": magnet.total_energy,
    "e_exchange": magnet.exchange_energy,
    "e_zeeman": magnet.zeeman_energy,
    "e_demag": magnet.demag_energy
}

# --- RUN THE SOLVER ---
output = world.timesolver.solve(timepoints, outputquantities)

# --- PLOT THE OUTPUT DATA ---
plt.subplot(211)
for key in ["mx", "my", "mz"]:
    plt.plot(output["time"], output[key], label=key)
plt.legend()

plt.subplot(212)
for key in ["e_total", "e_exchange", "e_zeeman", "e_demag"]:
    plt.plot(timepoints, output[key], label=key)
plt.legend()

plt.show()